**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Uncertainty in ML

A model that says '87%' should be right 87% of the time — most deep models aren't. Two sessions on measuring calibration and fixing it with the workhorse tools: temperature scaling and deep ensembles, with a hard look at what happens *off*-distribution.

## 1. Pre-requisites

- [Training Dynamics](./Training_Dynamics.ipynb) (we reuse its spiral testbed).
- [Kernel Methods](./Kernel_Methods.ipynb) S2 — GPs as the calibration gold standard.
- [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) S3 — the Bayesian frame.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

# noisy spirals again — genuine class overlap means genuine aleatoric uncertainty
def spirals(n=3000, noise=0.9, seed=0):
    r = np.random.default_rng(seed)
    t = np.linspace(0.5, 3*np.pi, n//2)
    X, y = [], []
    for cls, ph in [(0, 0.0), (1, np.pi)]:
        X.append(np.stack([t*np.cos(t+ph), t*np.sin(t+ph)], 1) + noise*r.standard_normal((n//2, 2)))
        y.append(np.full(n//2, cls))
    X = np.concatenate(X).astype(np.float32); y = np.concatenate(y).astype(np.int64)
    X = (X - X.mean(0)) / X.std(0)
    idx = r.permutation(n)
    return torch.from_numpy(X[idx]), torch.from_numpy(y[idx])

Xa, ya = spirals()
Xtr, ytr, Xte, yte = Xa[:2000], ya[:2000], Xa[2000:], ya[2000:]

def make_net(seed):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(2, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 2))

def fit(net, epochs=600):     # deliberately overtrained — watch the confidence outrun the accuracy
    opt = torch.optim.Adam(net.parameters(), lr=2e-3)
    for ep in range(epochs):
        for i in range(0, 2000, 200):
            opt.zero_grad()
            Fn.cross_entropy(net(Xtr[i:i+200]), ytr[i:i+200]).backward()
            opt.step()
    return net

---
### 🕐 Session 1 of 2 — *Calibration & Temperature Scaling* (~40 min)
**Goal:** measure whether confidences mean anything; fix miscalibration with one parameter.
**Builds on:** [Training Dynamics](./Training_Dynamics.ipynb). &nbsp; **Feeds into:** Session 2 (ensembles & OOD).

---

## 2. Does 87% Mean 87%?

💡 **Intuition.** Accuracy asks 'how often right?'; **calibration** asks 'when you say 87%, are you right 87% of the time?' The reliability diagram answers it: bin predictions by confidence, plot accuracy per bin against the diagonal. Deep nets trained to convergence sit *below* the diagonal — overconfident — because cross-entropy keeps rewarding sharper probabilities long after accuracy saturates. The embarrassingly effective fix: **temperature scaling** — divide the logits by one scalar $T$ fitted on validation data. It can't change any decision (argmax is $T$-invariant); it only makes the *confidence honest*.

In [2]:
net = fit(make_net(0))
with torch.no_grad():
    logits = net(Xte)
    conf, pred = Fn.softmax(logits, 1).max(1)
acc_overall = (pred == yte).float().mean()

def reliability(conf, pred, yv, bins=10):
    edges = np.linspace(0.5, 1.0, bins+1)
    accs, confs, ns = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf >= lo) & (conf < hi if hi < 1 else conf <= hi)
        if m.sum() > 4:
            accs.append((pred[m] == yv[m]).float().mean().item())
            confs.append(conf[m].mean().item()); ns.append(int(m.sum()))
    return np.array(confs), np.array(accs), np.array(ns)

def ece(conf, pred, yv):
    c, a, n_ = reliability(conf, pred, yv)
    return float(np.sum(n_ * np.abs(c - a)) / n_.sum())

print(f"accuracy {acc_overall:.1%}   ECE (expected calibration error) {ece(conf, pred, yte):.3f}")

accuracy 89.8%   ECE (expected calibration error) 0.041


In [3]:
# fit T on a validation split by minimizing NLL — one parameter, no retraining
T = torch.ones(1, requires_grad=True)
opt_t = torch.optim.LBFGS([T], lr=0.1, max_iter=50)
val_logits, val_y = logits[:500].detach(), yte[:500]
def closure():
    opt_t.zero_grad()
    loss = Fn.cross_entropy(val_logits / T.clamp(min=0.05), val_y)
    loss.backward(); return loss
opt_t.step(closure)

with torch.no_grad():
    conf_T, pred_T = Fn.softmax(logits[500:] / T, 1).max(1)
conf_raw, pred_raw = Fn.softmax(logits[500:], 1).max(1)

fig, ax = plt.subplots(figsize=(4.6, 4))
for name, (c_, p_) in [("raw", (conf_raw, pred_raw)), (f"T = {T.item():.2f}", (conf_T, pred_T))]:
    cc, aa, _ = reliability(c_, p_, yte[500:])
    ax.plot(cc, aa, "o-", label=f"{name} (ECE {ece(c_, p_, yte[500:]):.3f})")
ax.plot([0.5, 1], [0.5, 1], "k--", linewidth=0.8, label="perfect calibration")
ax.set_xlabel("stated confidence"); ax.set_ylabel("actual accuracy"); ax.legend(fontsize=8)
ax.set_title("reliability diagram: temperature drags the curve onto the diagonal")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2700996/40931829.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 2 — *Ensembles & the Out-of-Distribution Problem* (~40 min)
**Goal:** average independently-trained nets for better uncertainty; test where all bets are off.
**Builds on:** Session 1.

---

## 3. Deep Ensembles

💡 **Intuition.** Train the same architecture from $k$ different random seeds and *average the probabilities*. Where the data constrains the function, the members agree; where it doesn't, they disagree — and that **disagreement is an uncertainty signal** that single-model confidence simply doesn't carry. It's a crude Bayesian posterior ([Estimation Theory S3](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb)), cousin to the [particle filter](../Intro_Time_Series/Beyond_Kalman.ipynb): represent belief with samples. Cost: $k\times$ everything — and it remains the strongest practical baseline in the field.

In [4]:
def fit_boot(seed):
    torch.manual_seed(seed)
    idx = torch.randint(0, 2000, (2000,))            # bootstrap: each member sees a different resample
    net_b = make_net(seed)
    opt = torch.optim.Adam(net_b.parameters(), lr=2e-3)
    for ep in range(600):
        for i in range(0, 2000, 200):
            j = idx[i:i+200]
            opt.zero_grad(); Fn.cross_entropy(net_b(Xtr[j]), ytr[j]).backward(); opt.step()
    return net_b

members = [fit_boot(s) for s in range(5)]
with torch.no_grad():
    probs = torch.stack([Fn.softmax(m(Xte), 1) for m in members])
p_ens = probs.mean(0)
conf_e, pred_e = p_ens.max(1)
print(f"single model: acc {(pred == yte).float().mean():.1%}   ECE {ece(conf, pred, yte):.3f}")
print(f"5-ensemble:   acc {(pred_e == yte).float().mean():.1%}   ECE {ece(conf_e, pred_e, yte):.3f}")

single model: acc 89.8%   ECE 0.041
5-ensemble:   acc 89.4%   ECE 0.041


## 4. Off the Map

💡 **Intuition.** The dirty secret of every confidence score: it is only meaningful **on the distribution the model was trained on**. Show the network a point from a different world and softmax still prints a confident number — softmax *must* sum to one; it has no 'none of the above'. Ensemble disagreement (predictive entropy) at least *rises* off-distribution. Compare both on points far outside the spiral:

In [5]:
# scan the plane: single-model confidence vs ensemble entropy
g = np.linspace(-4, 4, 200)
GX, GY = np.meshgrid(g, g)
grid = torch.tensor(np.stack([GX.ravel(), GY.ravel()], 1), dtype=torch.float32)
with torch.no_grad():
    p_single = Fn.softmax(net(grid), 1).max(1).values.reshape(GX.shape)
    pg = torch.stack([Fn.softmax(m(grid), 1) for m in members]).mean(0)
    ent = (-pg * pg.clamp(min=1e-9).log()).sum(1).reshape(GX.shape)

fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.6))
im0 = axes[0].contourf(GX, GY, p_single, levels=20, cmap="RdYlGn")
axes[0].set_title("single net max-softmax:\nCONFIDENT even far from any data")
im1 = axes[1].contourf(GX, GY, ent, levels=20, cmap="RdYlGn_r")
axes[1].set_title("ensemble predictive entropy:\nuncertainty rises off the spiral")
for ax in axes:
    ax.scatter(*Xtr[:500].T, s=1, c="k", alpha=0.3)
plt.colorbar(im0, ax=axes[0]); plt.colorbar(im1, ax=axes[1])
plt.tight_layout(); plt.show()

# find the off-map points the ensemble actually flags (and admit the ones it doesn't)
radius = np.hypot(GX, GY)
off_map = radius > 3.0
ent_np, ps_np = ent.numpy(), p_single.numpy()
flagged = (ent_np > 0.35) & off_map
fooled  = (ent_np < 0.05) & off_map & (ps_np > 0.99)
print(f"off-map area flagged by ensemble entropy (>0.35): {flagged.sum()/off_map.sum():.0%}")
print(f"off-map area where ensemble AND single net are both confidently wrong-headed: {fooled.sum()/off_map.sum():.0%}")
iy, ix = np.unravel_index(np.argsort(-ent_np*off_map, axis=None)[:3], ent_np.shape)
far = torch.tensor(np.stack([GX[iy, ix], GY[iy, ix]], 1), dtype=torch.float32)
with torch.no_grad():
    sm = Fn.softmax(net(far), 1).max(1).values
    pe = torch.stack([Fn.softmax(m(far), 1) for m in members]).mean(0)
    he = (-pe * pe.clamp(min=1e-9).log()).sum(1)
for i in range(3):
    print(f"flagged point ({far[i,0]:+.1f}, {far[i,1]:+.1f}): single-net confidence {sm[i]:.1%}   ensemble entropy {he[i]:.2f} (max {np.log(2):.2f})")

off-map area flagged by ensemble entropy (>0.35): 9%
off-map area where ensemble AND single net are both confidently wrong-headed: 90%
flagged point (+3.1, +1.6): single-net confidence 100.0%   ensemble entropy 0.69 (max 0.69)
flagged point (+3.9, +2.3): single-net confidence 100.0%   ensemble entropy 0.69 (max 0.69)
flagged point (+3.7, +2.1): single-net confidence 100.0%   ensemble entropy 0.69 (max 0.69)


/tmp/ipykernel_2700996/1850092308.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 5. Conclusion

Calibrate before you trust (temperature is nearly free); ensemble when the stakes justify 5×; and treat *all* confidences as conditional on being on-distribution — detecting 'off the map' is its own problem, and disagreement is your first tool — but as the area numbers show, it flags only *part* of the off-map world: ensembles mitigate OOD overconfidence, they do not solve it. The [GP](./Kernel_Methods.ipynb) remains the standard these methods chase.

---
## Where next

- [Kernel Methods](./Kernel_Methods.ipynb) S2 — calibrated uncertainty by construction.
- [Beyond Kalman](../Intro_Time_Series/Beyond_Kalman.ipynb) — belief-as-samples, the filtering version.
- [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) — what 'well-calibrated' means formally.